# Validación de la importación del baseline fuente

Este notebook es la validación interactiva principal para comprobar que los CSV de `database/source` se importan correctamente al modelo ORM.

La validación se enfoca en las tablas dimensionales `dm_*` y en la tabla de hechos `fact_indicator_response`.

Cada escenario usa una base SQLite temporal bajo `.test_tmp` y libera el engine al final para evitar archivos bloqueados en Windows.

## 1. Preparación

La siguiente celda encuentra la raíz del proyecto, importa el loader del baseline y define helpers para contar filas, leer los CSV fuente y validar relaciones entre tablas.

In [ ]:
import csv
import shutil
import sys
import tempfile
from datetime import date
from pathlib import Path
from pprint import pprint

from sqlalchemy import distinct, func, select


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "database").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("No se encontro la raiz del proyecto con database/ y data/.")


ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from database.import_source_baseline import load_source_baseline  # noqa: E402
from database.repositories import (  # noqa: E402
    get_latest_indicator_responses,
    get_municipality_completion_statistics,
    get_national_statistics,
)
from database.models import (  # noqa: E402
    DMAxis,
    DMIndicator,
    DMMunicipality,
    DMMunicipalityDiversifiedService,
    DMService,
    DMStage,
    FactIndicatorResponse,
)
from database.session import get_engine, session_scope  # noqa: E402


SOURCE_DIR = ROOT / "database" / "source"
TEMP_PARENT = ROOT / ".test_tmp"
BASELINE_END_DATE = date(2025, 12, 31)
RESULTS_CSV = SOURCE_DIR / "igsm_2025_results_long.csv"
DICTIONARY_CSV = SOURCE_DIR / "dictionary.csv"

EXPECTED = {
    "source_result_rows": 13356,
    "source_blank_values": 4516,
    "source_nonblank_values": 8840,
    "dm_municipality": 84,
    "dm_municipality_diversified_service": 97,
    "dm_axis": 3,
    "dm_service": 10,
    "dm_stage": 3,
    "service_stage_pairs": 30,
    "dm_indicator": 159,
    "fact_indicator_response": 8840,
    "fact_distinct_indicators": 146,
}


def database_url(tmpdir: str | Path) -> str:
    db_path = Path(tmpdir) / "baseline.sqlite3"
    return f"sqlite:///{db_path.as_posix()}"


def count(session, model) -> int:
    return int(session.scalar(select(func.count()).select_from(model)) or 0)


def distinct_count(session, column) -> int:
    return int(session.scalar(select(func.count(distinct(column)))) or 0)


def dispose(url: str) -> None:
    get_engine(url).dispose()


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return [{key: value or "" for key, value in row.items()} for row in csv.DictReader(file)]


def rewrite_first_data_row(path: Path, column: str, value: str) -> None:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
        fieldnames = list(rows[0].keys())

    row_index = 0
    if column == "Valor":
        row_index = next(index for index, row in enumerate(rows) if row["Valor"].strip())
    rows[row_index][column] = value

    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


TEMP_PARENT.mkdir(exist_ok=True)
print(f"Project root: {ROOT}")
print(f"Source CSV directory: {SOURCE_DIR}")
print(f"Temporary test parent: {TEMP_PARENT}")

## 2. Validar los CSV fuente

Antes de tocar la base de datos, esta celda valida que los archivos fuente tengan el volumen esperado: 159 indicadores en el diccionario, 13,356 filas de resultados, 4,516 valores vacíos y 8,840 valores numéricos importables.

In [ ]:
dictionary_rows = read_csv_rows(DICTIONARY_CSV)
result_rows = read_csv_rows(RESULTS_CSV)
blank_values = [row for row in result_rows if not row["Valor"].strip()]
nonblank_values = [row for row in result_rows if row["Valor"].strip()]
dictionary_codes = {row["Código"].strip() for row in dictionary_rows}
result_codes = {row["Código"].strip() for row in result_rows}
nonblank_result_codes = {row["Código"].strip() for row in nonblank_values}

source_counts = {
    "dictionary_indicators": len(dictionary_rows),
    "unique_dictionary_codes": len(dictionary_codes),
    "source_result_rows": len(result_rows),
    "source_blank_values": len(blank_values),
    "source_nonblank_values": len(nonblank_values),
    "unique_result_codes": len(result_codes),
    "unique_nonblank_result_codes": len(nonblank_result_codes),
}

print("Conteos de los CSV fuente:")
pprint(source_counts)

assert source_counts["dictionary_indicators"] == EXPECTED["dm_indicator"]
assert source_counts["unique_dictionary_codes"] == EXPECTED["dm_indicator"]
assert source_counts["source_result_rows"] == EXPECTED["source_result_rows"]
assert source_counts["source_blank_values"] == EXPECTED["source_blank_values"]
assert source_counts["source_nonblank_values"] == EXPECTED["source_nonblank_values"]
assert source_counts["unique_nonblank_result_codes"] == EXPECTED["fact_distinct_indicators"]
assert result_codes.issubset(dictionary_codes)

## 3. Importar el baseline y validar `dm_*` + `fact_*`

Esta es la validación principal. Importa el baseline a una SQLite temporal y comprueba que las dimensiones, las 30 combinaciones servicio-etapa y la tabla de hechos tengan los conteos esperados, llaves unicas, relaciones válidas y valores de hechos dentro del rango esperado.

In [ ]:
with tempfile.TemporaryDirectory(dir=TEMP_PARENT) as tmpdir:
    url = database_url(tmpdir)
    try:
        summary = load_source_baseline(database_url=url)

        with session_scope(url) as session:
            table_counts = {
                "dm_municipality": count(session, DMMunicipality),
                "dm_municipality_diversified_service": count(session, DMMunicipalityDiversifiedService),
                "dm_axis": count(session, DMAxis),
                "dm_service": count(session, DMService),
                "dm_stage": count(session, DMStage),
                "dm_indicator": count(session, DMIndicator),
                "fact_indicator_response": count(session, FactIndicatorResponse),
            }
            unique_counts = {
                "municipality_codes": distinct_count(session, DMMunicipality.code),
                "axis_names": distinct_count(session, DMAxis.name),
                "service_codes": distinct_count(session, DMService.service_code),
                "indicator_codes": distinct_count(session, DMIndicator.code),
                "fact_municipalities": distinct_count(session, FactIndicatorResponse.municipality_id),
                "fact_indicators": distinct_count(session, FactIndicatorResponse.indicator_id),
            }
            service_stage_pair_rows = select(DMIndicator.service_id, DMIndicator.stage_id).distinct().subquery()

            national_stats = get_national_statistics(end_date=BASELINE_END_DATE, session=session)
            first_municipality = session.execute(
                select(DMMunicipality).order_by(DMMunicipality.code)
            ).scalars().first()
            municipality_stats = get_municipality_completion_statistics(
                first_municipality.code,
                end_date=BASELINE_END_DATE,
                session=session,
            )
            latest_responses = get_latest_indicator_responses(end_date=BASELINE_END_DATE, session=session)
            municipality_response_count = int(
                session.scalar(
                    select(func.count(FactIndicatorResponse.indicator_id.distinct())).where(
                        FactIndicatorResponse.municipality_id == first_municipality.municipality_id
                    )
                )
                or 0
            )
            relationship_counts = {
                "services_with_axis": int(session.scalar(select(func.count()).select_from(DMService).join(DMAxis)) or 0),
                "service_stage_pairs": int(session.scalar(select(func.count()).select_from(service_stage_pair_rows)) or 0),
                "indicators_with_service": int(session.scalar(select(func.count()).select_from(DMIndicator).join(DMService)) or 0),
                "indicators_with_stage": int(session.scalar(select(func.count()).select_from(DMIndicator).join(DMStage)) or 0),
                "facts_with_municipality": int(session.scalar(select(func.count()).select_from(FactIndicatorResponse).join(DMMunicipality)) or 0),
                "facts_with_indicator": int(session.scalar(select(func.count()).select_from(FactIndicatorResponse).join(DMIndicator)) or 0),
                "diversified_with_municipality": int(
                    session.scalar(
                        select(func.count()).select_from(DMMunicipalityDiversifiedService).join(DMMunicipality)
                    )
                    or 0
                ),
                "diversified_with_service": int(
                    session.scalar(select(func.count()).select_from(DMMunicipalityDiversifiedService).join(DMService))
                    or 0
                ),
            }
            stage_names = set(session.execute(select(DMStage.name).distinct()).scalars())
            fact_min, fact_max = session.execute(
                select(func.min(FactIndicatorResponse.value), func.max(FactIndicatorResponse.value))
            ).one()
    finally:
        dispose(url)

summary_for_validation = {
    key: summary[key]
    for key in [
        "period",
        "skipped",
        "source_indicators",
        "source_values_total",
        "blank_values_skipped",
        "nonblank_values",
        "source_municipalities",
        "axes",
        "services",
        "stages",
        "indicators",
        "municipalities",
        "facts_inserted",
    ]
}

print("Resumen del importador para dm/fact:")
pprint(summary_for_validation)
print("\nConteos por tabla:")
pprint(table_counts)
print("\nConteos unicos:")
pprint(unique_counts)
print("\nConteos con relaciones validas:")
pprint(relationship_counts)
print(f"\nEtapas encontradas: {sorted(stage_names)}")
print(f"Rango de valores fact: {fact_min} - {fact_max}")
print("\nCompletitud nacional:")
pprint(national_stats)
print("\nCompletitud municipal de muestra:")
pprint(municipality_stats)

assert summary["skipped"] is False
assert summary["source_indicators"] == EXPECTED["dm_indicator"]
assert summary["source_values_total"] == EXPECTED["source_result_rows"]
assert summary["blank_values_skipped"] == EXPECTED["source_blank_values"]
assert summary["nonblank_values"] == EXPECTED["source_nonblank_values"]
assert summary["facts_inserted"] == EXPECTED["fact_indicator_response"]
assert table_counts == {key: EXPECTED[key] for key in table_counts}
assert unique_counts == {
    "municipality_codes": EXPECTED["dm_municipality"],
    "axis_names": EXPECTED["dm_axis"],
    "service_codes": EXPECTED["dm_service"],
    "indicator_codes": EXPECTED["dm_indicator"],
    "fact_municipalities": EXPECTED["dm_municipality"],
    "fact_indicators": EXPECTED["fact_distinct_indicators"],
}
assert relationship_counts == {
    "services_with_axis": EXPECTED["dm_service"],
    "service_stage_pairs": EXPECTED["service_stage_pairs"],
    "indicators_with_service": EXPECTED["dm_indicator"],
    "indicators_with_stage": EXPECTED["dm_indicator"],
    "facts_with_municipality": EXPECTED["fact_indicator_response"],
    "facts_with_indicator": EXPECTED["fact_indicator_response"],
    "diversified_with_municipality": EXPECTED["dm_municipality_diversified_service"],
    "diversified_with_service": EXPECTED["dm_municipality_diversified_service"],
}
assert stage_names == {"Planificación", "Ejecución", "Evaluación"}
assert fact_min >= 0.0
assert fact_max <= 1.0
assert national_stats["end_date"] == BASELINE_END_DATE.isoformat()
assert national_stats["total_municipalidades"] == EXPECTED["dm_municipality"]
assert national_stats["total_indicadores"] == EXPECTED["dm_indicator"]
assert national_stats["respuestas_esperadas"] == EXPECTED["dm_municipality"] * EXPECTED["dm_indicator"]
assert national_stats["respuestas_recibidas"] == EXPECTED["fact_indicator_response"]
assert len(latest_responses) == EXPECTED["fact_indicator_response"]
assert national_stats["pct_completitud"] == round(
    EXPECTED["fact_indicator_response"] / national_stats["respuestas_esperadas"] * 100,
    2,
)
assert municipality_stats["codigo"] == first_municipality.code
assert municipality_stats["respuestas_esperadas"] == EXPECTED["dm_indicator"]
assert municipality_stats["respuestas_recibidas"] == municipality_response_count


## 4. Importación repetida no debe duplicar dimensiones ni hechos

Al ejecutar el import dos veces para el mismo periodo, los conteos de `dm_*` y `fact_indicator_response` deben quedar iguales. Esta celda comprueba que el resultado final de dimensiones y hechos siga siendo estable.

In [ ]:
with tempfile.TemporaryDirectory(dir=TEMP_PARENT) as tmpdir:
    url = database_url(tmpdir)
    try:
        first_summary = load_source_baseline(database_url=url)
        second_summary = load_source_baseline(database_url=url)

        with session_scope(url) as session:
            counts_after_second_import = {
                "dm_municipality": count(session, DMMunicipality),
                "dm_municipality_diversified_service": count(session, DMMunicipalityDiversifiedService),
                "dm_axis": count(session, DMAxis),
                "dm_service": count(session, DMService),
                "dm_stage": count(session, DMStage),
                "dm_indicator": count(session, DMIndicator),
                "fact_indicator_response": count(session, FactIndicatorResponse),
            }
    finally:
        dispose(url)

first_summary_for_validation = {
    "facts_inserted": first_summary["facts_inserted"],
    "facts_deleted": first_summary["facts_deleted"],
}
second_summary_for_validation = {
    "facts_inserted": second_summary["facts_inserted"],
    "facts_deleted": second_summary["facts_deleted"],
}

print("Primer import dm/fact:")
pprint(first_summary_for_validation)
print("\nSegundo import dm/fact:")
pprint(second_summary_for_validation)
print("\nConteos despues del segundo import:")
pprint(counts_after_second_import)

assert first_summary["facts_inserted"] == EXPECTED["fact_indicator_response"]
assert second_summary["facts_inserted"] == EXPECTED["fact_indicator_response"]
assert second_summary["facts_deleted"] == EXPECTED["fact_indicator_response"]
assert counts_after_second_import == {key: EXPECTED[key] for key in counts_after_second_import}

## 5. Filas inválidas deben fallar claramente

El importador debe rechazar datos fuente mal formados antes de poblar la base. Esta celda copia los CSV a una carpeta temporal, altera una columna por caso y confirma que aparezca el mensaje de error esperado.

In [ ]:
invalid_cases = [
    ("Código", "9.9.9.9", "Unknown indicator code"),
    ("Valor", "not-a-number", "Nonnumeric Valor"),
    ("Cantón", "Cantón Fantasma", "Unknown municipality"),
]

observed_errors = []

for column, value, expected_message in invalid_cases:
    with tempfile.TemporaryDirectory(dir=TEMP_PARENT) as tmpdir:
        url = database_url(tmpdir)
        source_dir = Path(tmpdir) / "source"
        shutil.copytree(SOURCE_DIR, source_dir)
        rewrite_first_data_row(source_dir / "igsm_2025_results_long.csv", column, value)

        try:
            load_source_baseline(source_dir=source_dir, database_url=url)
        except ValueError as exc:
            message = str(exc)
            observed_errors.append((column, message))
            assert expected_message in message
        else:
            raise AssertionError(f"Se esperaba ValueError con {expected_message!r} para columna {column!r}")
        finally:
            dispose(url)

print("Errores esperados observados:")
pprint(observed_errors)

assert len(observed_errors) == len(invalid_cases)

## Resultado esperado

Si todas las celdas corren sin `AssertionError`, el baseline fue validado correctamente en las tablas `dm_*`, en `fact_indicator_response` y en los endpoints de completitud del ORM.
